# Korpora vergleichen

Skript 01 endete mit einem Problem: Reine Häufigkeitstabellen werden von Wörtern dominiert, die *überall* häufig sind, und sagen daher wenig darüber aus, was eine Quelle von einer anderen unterscheidet. In diesem Skript geht es um Maße, die für den Vergleich gemacht sind.

Am Ende dieses Skripts kannst du:

- ein Korpus als **Dokument-Term-Matrix** darstellen;
- **TF-IDF** von Hand und mit scikit-learn berechnen und erklären, was es belohnt;
- eine **Keyness**-Analyse mit Log-Likelihood und Log Ratio durchführen und erklären, warum man beide braucht;
- **Kollokationen** mit Pointwise Mutual Information finden;
- jeden statistischen Befund mit einer **Konkordanz** am Text überprüfen;
- Begriffe **im Zeitverlauf** verfolgen.

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Standardbibliothek
import math
from collections import Counter

# Visualisierung und Datentransformation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Pakete für die Sprachverarbeitung und die Dokument-Term-Matrix
import spacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Hilfsfunktionen des Kurses
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Daten

Wir laden das Korpus und das deutsche spaCy-Modell. Die Schlagzeilen hat bereits Skript 02 mit spaCy annotiert; `tm.parse_corpus` liest das Ergebnis aus dem Cache, statt die Texte erneut zu parsen.

In [ ]:
# Lade das Korpus und das spaCy-Modell
corpus = tm.load_corpus()
nlp = spacy.load("de_core_news_md")

# Lade die annotierten Schlagzeilen (im Cache von Skript 02)
docs = tm.parse_corpus(nlp, corpus["title"].tolist(), name="titles")

Wir setzen beim Ergebnis von Skript 02 an: Lemmata der Inhaltswörter, in Kleinschreibung (Casefolding). Jede Methode in diesem Skript arbeitet mit Listen von Tokens, daher lässt sich die Darstellung austauschen (rohe Tokens, nur Substantive), ohne sonst etwas zu ändern.

In [ ]:
# Definiere, was als Inhaltswort zählt: keine Satzzeichen, Zahlen, Symbole oder Stoppwörter
def is_content(token) -> bool:
    return token.pos_ not in {"PUNCT", "NUM", "SPACE", "SYM", "X"} and not token.is_stop and any(
        ch.isalpha() for ch in token.text
    )


corpus["lemmas"] = [[t.lemma_.casefold() for t in d if is_content(t)] for d in docs]
corpus["doc_id"] = np.arange(len(corpus))  # Position in `docs`, bleibt beim Filtern erhalten
corpus[["outlet", "title", "lemmas"]].head(3)

### Gleiches mit Gleichem vergleichen

Das Korpus enthält zwei Zeitungen aus zwei Ländern, aber in sehr verschiedener Zusammensetzung: Skript 01 hat gezeigt, dass die SZ-Daten viel Sport enthalten und die Standard-Daten ein großes Web-Ressort. Ein Vergleich der beiden Zeitungen als Ganzes würde deshalb vor allem wiederentdecken, dass Sport nicht Technik ist. Aufschlussreich sind Vergleiche, die die **Art des Inhalts festhalten** und nur das Land variieren. Dafür ordnet `textmining.py` die Ressorts beider Redaktionen gemeinsamen Ressortgruppen zu (Spalte `section_group`):

| Ressortgruppe | Der Standard (AT) | Süddeutsche Zeitung (DE) |
|---|---|---|
| Politik | International, Inland | politik |
| Wirtschaft | Wirtschaft | wirtschaft, geld |
| Sport | Sport | sport |
| Panorama | Panorama | panorama |
| Digital | Web | digital |

Dieses Skript konzentriert sich auf die **Politik** - Der Standard gegen Süddeutsche Zeitung, jeweils nur die politischen Ressorts - und überlässt die anderen Gruppen den Übungen.

In [ ]:
# Wähle die Politik-Ressorts beider Zeitungen aus
news = corpus[corpus["section_group"] == "Politik"].reset_index(drop=True)
news.groupby(["outlet", "section"]).size()

## Die Dokument-Term-Matrix

Fast jede statistische Textmethode beginnt damit, das Korpus in eine Matrix zu verwandeln: eine **Zeile je Dokument**, eine **Spalte je Term**, und in jeder Zelle die Anzahl, wie oft der Term in diesem Dokument vorkommt. Die Wortreihenfolge geht dabei verloren, deshalb spricht man von einer **Bag-of-Words**-Darstellung.

Der `CountVectorizer` von scikit-learn erstellt diese Matrix. Da unsere Tokens bereits aufbereitet sind, übergeben wir als Analyzer eine Identitätsfunktion. So tokenisiert der Vectorizer nicht selbst, und unsere Pipeline bleibt nachvollziehbar.

In [ ]:
# Gib die Token-Liste unverändert zurück: Die Tokenisierung haben wir bereits erledigt
def identity(tokens):
    return tokens


# Erstelle die Dokument-Term-Matrix
counter = CountVectorizer(analyzer=identity, min_df=1)
dtm = counter.fit_transform(news["lemmas"])
vocab = counter.get_feature_names_out()

print(f"Form: {dtm.shape[0]:,} Dokumente x {dtm.shape[1]:,} Terme")
print(f"Zellen ungleich null: {dtm.nnz:,} von {dtm.shape[0] * dtm.shape[1]:,} ({dtm.nnz / np.prod(dtm.shape):.2%})")

Die Matrix besteht fast nur aus Nullen - jede Schlagzeile verwendet eine Handvoll der Tausenden Terme im Vokabular. Deshalb wird sie als **dünnbesetzte** Matrix (engl. *sparse*) gespeichert, die nur die Zellen ungleich null festhält. Eine vollbesetzte Version wäre hunderte Male größer, ohne irgendeinen Nutzen.

## TF-IDF

**TF-IDF** gewichtet jede Zelle mit zwei Faktoren:

- **Termfrequenz** (tf) - wie oft der Term in diesem Dokument vorkommt;
- **inverse Dokumentfrequenz** (idf) - wie *selten* der Term über alle Dokumente hinweg ist: $\text{idf}(t) = \ln\frac{1 + N}{1 + \text{df}(t)} + 1$, wobei *N* die Anzahl der Dokumente ist und df(*t*) die Anzahl der Dokumente, die *t* enthalten (das ist die geglättete Variante von scikit-learn).

Ein Term, der in jedem Dokument vorkommt, erhält das minimale Gewicht, ein Term in einem einzigen Dokument das maximale. Das Produkt belohnt Terme, die *hier* häufig und *anderswo* selten sind. Berechnen wir idf für zwei Terme von Hand, wird die Größenordnung greifbar:

In [ ]:
# Berechne die Dokumentfrequenz jedes Terms
df = np.asarray((dtm > 0).sum(axis=0)).ravel()
N = dtm.shape[0]

# Berechne idf für einen häufigen und einen seltenen Term
for term in ["merkel", news["lemmas"].explode().value_counts().index[-1]]:
    i = np.where(vocab == term)[0][0]
    print(f"{term:>20}: in {df[i]:4} von {N} Schlagzeilen, idf = {math.log((1 + N) / (1 + df[i])) + 1:.2f}")

Nun mit dem `TfidfVectorizer`, der zusätzlich den Vektor jedes Dokuments auf Länge eins normalisiert, damit lange und kurze Dokumente vergleichbar sind. `min_df=3` entfernt Terme, die in weniger als drei Schlagzeilen vorkommen: Skript 01 hat gezeigt, dass der Großteil des Vokabulars so selten ist, und Einzelvorkommen liefern keine Evidenz für einen Vergleich.

In [ ]:
# Erstelle die TF-IDF-Matrix
tfidf = TfidfVectorizer(analyzer=identity, min_df=3)
X = tfidf.fit_transform(news["lemmas"])
terms = tfidf.get_feature_names_out()
print(f"{len(terms):,} von {len(vocab):,} Termen behalten")


# Finde die Terme mit dem höchsten mittleren TF-IDF-Wert in einer Teilmenge der Dokumente
def characteristic_terms(mask, n=15):
    scores = np.asarray(X[mask].mean(axis=0)).ravel()
    top = scores.argsort()[::-1][:n]
    return [f"{terms[i]} ({scores[i]:.3f})" for i in top]


pd.DataFrame({o: characteristic_terms((news["outlet"] == o).to_numpy()) for o in tm.OUTLETS.index})

Der mittlere TF-IDF-Wert je Zeitung ist ein brauchbarer erster Blick, aber kein *Vergleich*: Ein Term kann für beide Zeitungen hoch abschneiden, und die Tabelle bietet keine Möglichkeit zu sagen, ob ein Unterschied zwischen ihnen größer ist als der Zufall. Dafür brauchen wir einen Test, der für genau diese Frage gemacht ist.

## Keyness

**Keyness** fragt Term für Term: Kommt dieses Wort in Korpus A häufiger vor, als wir erwarten würden, wenn A und B aus derselben Sprache gezogen wären? Das ist die Standardmethode der Korpuslinguistik für genau die Frage "Was ist an diesem Text charakteristisch?".

Für jeden Term haben wir vier Zahlen: seine Häufigkeit *a* im Zielkorpus und *b* im Referenzkorpus sowie die Korpusgrößen *c* und *d* in Tokens.

Zwei Statistiken beantworten zwei verschiedene Fragen:

- **Log-Likelihood (G²)** - *Wie sicher sind wir, dass es einen Unterschied gibt?* Sie vergleicht die beobachteten Häufigkeiten mit jenen, die zu erwarten wären, wenn der Term gleichmäßig verteilt wäre, und wächst sowohl mit der Größe des Unterschieds als auch mit der Menge an Evidenz. Werte über 15,13 entsprechen p < 0,0001 bei einem Freiheitsgrad.
- **Log Ratio** - *Wie groß ist der Unterschied?* Sie ist der binäre Logarithmus des Verhältnisses der relativen Häufigkeiten: 1 bedeutet doppelt so häufig, 2 viermal so häufig und so weiter.

Wir brauchen beide, weil sie absichtlich voneinander abweichen. Ein sehr häufiger Term kann aus einem kleinen Verhältnis ein riesiges G² erzielen, und ein seltener Term kann aus einer Handvoll Vorkommen, die nichts beweisen, ein großes Verhältnis erzielen.

In [ ]:
# Berechne Log-Likelihood (G2) und Log Ratio für jeden Term
def keyness(target: list[list[str]], reference: list[list[str]], min_freq: int = 5) -> pd.DataFrame:
    ta = Counter(t for doc in target for t in doc)
    rb = Counter(t for doc in reference for t in doc)
    c, d = sum(ta.values()), sum(rb.values())
    rows = []
    for term in set(ta) | set(rb):
        a, b = ta[term], rb[term]
        if a + b < min_freq:
            continue
        e1 = c * (a + b) / (c + d)
        e2 = d * (a + b) / (c + d)
        g2 = 2 * ((a * math.log(a / e1) if a else 0) + (b * math.log(b / e2) if b else 0))
        # Log Ratio nach Hardie, mit 0,5 addiert, damit eine Häufigkeit von null endlich bleibt.
        lr = math.log2(((a + 0.5) / c) / ((b + 0.5) / d))
        rows.append((term, a, b, g2 if lr > 0 else -g2, lr))
    return pd.DataFrame(rows, columns=["term", "target", "reference", "G2", "log_ratio"])


# Vergleiche Der Standard (Zielkorpus) mit der Süddeutschen Zeitung (Referenzkorpus)
key = keyness(
    news.loc[news["outlet"] == "Der Standard", "lemmas"].tolist(),
    news.loc[news["outlet"] == "Süddeutsche Zeitung", "lemmas"].tolist(),
)
print(f"{len(key):,} Terme getestet; {(key['G2'].abs() > 15.13).sum()} erreichen p < 0.0001")

G² ist hier mit Vorzeichen versehen: positiv für Terme, die in Der Standard überdurchschnittlich oft vorkommen, negativ für Terme, die in der Süddeutschen Zeitung überdurchschnittlich oft vorkommen. Weil Keyness mit relativen Häufigkeiten rechnet, spielt es keine Rolle, dass das SZ-Teilkorpus mehr als dreimal so groß ist.

In [ ]:
# Zeige die stärksten Schlüsselbegriffe beider Zeitungen nebeneinander
cols = ["term", "target", "reference", "G2", "log_ratio"]
pd.concat(
    {
        "Schlüsselbegriffe Der Standard": key.nlargest(15, "G2")[cols].reset_index(drop=True),
        "Schlüsselbegriffe Süddeutsche Zeitung": key.nsmallest(15, "G2")[cols].reset_index(drop=True),
    },
    axis=1,
).round(1)

Lies die beiden Listen gegeneinander und ordne, was du siehst, vier Arten von Schlüsselbegriffen zu. Sie bedeuten sehr Verschiedenes:

1. **Innenpolitik** - die eigenen Parteien, Politikerinnen und Politiker, Orte und Themen jedes Landes (*SPÖ*, *FPÖ*, *ÖVP*, *Mindestsicherung*, *Oberösterreich*, die Präsidentschaftskandidaten *Griss* und *Van der Bellen* gegen *Merkel*, *AfD*, *SPD*, *Seehofer*, *Schäuble*). Das ist ein Unterschied in der *Berichterstattung*, und er bestätigt die Parteizählungen aus Skript 02 aus einem völlig anderen Blickwinkel.
2. **Hausformate** - wiederkehrende Rubriken, die in den Schlagzeilen sichtbar werden: Bei der SZ stehen *Presseschau* und *Kolumne* weit oben, weil sie als Dachzeilen vor vielen Titeln stehen. Das sind Unterschiede darin, *wie eine Zeitung ihre Texte etikettiert*, nicht darin, was berichtenswert ist.
3. **Hausstil** - dieselbe Sache, anders benannt. Österreichisches und bundesdeutsches Deutsch unterscheiden sich im Wortschatz, und Redaktionen haben eigene Schreibweisen für fremde Wörter. Das sind Unterschiede in der *Sprache*, überhaupt nicht im Inhalt.
4. **Zuschnitt der Ressorts** - der Schlüsselbegriff der SZ mit der größten Evidenz ist *Flüchtling*. Das heißt aber nicht, dass Der Standard weniger über Flüchtlinge geschrieben hätte: Er legt diese Berichte großteils im Ressort *Panorama* ab, das hier gar nicht mitgezählt wird. Auch die *Panama Papers* stehen bei Der Standard im Ressort *Wirtschaft*. Selbst ein Vergleich innerhalb einer Ressortgruppe hält die Art des Inhalts also nur ungefähr fest.

Eine Keyness-Liste kann diese Arten nicht unterscheiden. Du kannst es, durch Lesen. Die dritte Art lässt sich leicht direkt bestätigen:

In [ ]:
# Zähle je Zeitung die Dokumente (Schlagzeile plus Teaser) mit unterschiedlichen Schreibweisen derselben Sache
conventions = {
    "Asylwerber": r"Asylwerber", "Asylbewerber": r"Asylbewerber",
    "Jänner": r"\bJänner\b", "Januar": r"\bJanuar\b",
    "Jihadisten": r"Jihadist", "Dschihadisten": r"Dschihadist",
    "Spital": r"Spital", "Krankenhaus": r"Krankenhaus",
}
pd.DataFrame(
    {name: news.loc[news["text"].str.contains(rx), "outlet"].value_counts() for name, rx in conventions.items()}
).fillna(0).astype(int)

Die ersten drei Paare sind praktisch perfekt getrennt: *Asylwerber*, *Jänner* und *Jihadisten* schreibt nur Der Standard, *Asylbewerber*, *Januar* und *Dschihadisten* nur die SZ. Beim letzten Paar ist die Trennung einseitig: Das österreichische *Spital* kommt nur bei Der Standard vor, das *Krankenhaus* verwenden beide. Eine naive Lesart einer Keyness-Tabelle hätte ergeben, dass nur die SZ über Asylbewerber berichtet; tatsächlich tun es beide, nur unter verschiedenen Namen. Soll ein Vergleich von *Themen* handeln, sollten solche Konventionen vor dem Zählen normalisiert werden (indem man *Asylwerber* und *Asylbewerber* auf einen Term abbildet).

### Signifikanz ist nicht Effektgröße

Stellen wir beide Statistiken gemeinsam dar, sehen wir, warum es in die Irre führt, nur eine davon anzugeben:

In [ ]:
# Trage für jeden Term die Effektgröße (Log Ratio) gegen die Evidenz (|G2|) auf
fig, ax = plt.subplots(figsize=(8, 5))
sig = key["G2"].abs() > 15.13
ax.scatter(key.loc[~sig, "log_ratio"], key.loc[~sig, "G2"].abs(), s=8, color="0.75", label="nicht signifikant")

# Färbe die signifikanten Terme je Zeitung ein und beschrifte die stärksten
for outlet, sign in [("Der Standard", 1), ("Süddeutsche Zeitung", -1)]:
    colour = tm.OUTLET_COLOURS[outlet]
    m = sig & (np.sign(key["G2"]) == sign)
    ax.scatter(key.loc[m, "log_ratio"], key.loc[m, "G2"].abs(), s=14, color=colour, label=f"Schlüsselbegriffe {outlet}")
    for _, r in key[m].assign(a=key["G2"].abs()).nlargest(7, "a").iterrows():
        ax.annotate(r["term"], (r["log_ratio"], abs(r["G2"])), fontsize=8, xytext=(3, 2), textcoords="offset points")

# Zeichne die Signifikanzschwelle ein
ax.axhline(15.13, color="0.5", lw=0.8, ls="--")
ax.set(xlabel="Log Ratio  (Effektgröße: positiv = häufiger in Der Standard)", ylabel="|G²|  (Evidenz)",
       title="Keyness: Evidenz gegen Effektgröße")
ax.legend(frameon=False, fontsize=8)
plt.show()

Punkte weit oben sind die gesicherten Unterschiede. Punkte weit links oder rechts, aber weit unten, sind große Verhältnisse mit wenig Evidenz: oft Terme, die nur eine Handvoll Mal vorkommen, alle in einer Zeitung. Punkte weit oben, aber nahe der Mitte, sind häufige Terme mit einem echten, aber bescheidenen Unterschied. Ein guter Keyness-Bericht nennt für jeden besprochenen Term beide Zahlen.

Noch eine Warnung. Hunderte Terme wurden gleichzeitig getestet, daher braucht es eine strenge Schwelle, damit zufällige Treffer selten bleiben - deshalb erreichen bei einem Korpus von Schlagzeilen dieser Größe nur wenige Terme p < 0,0001. Keyness-Listen liest man am besten als **Hinweise, die zu prüfen sind** - und genau dafür ist die Konkordanz im Abschnitt "Konkordanzen: zurück zum Text" da.

## Kollokationen

Eine **Kollokation** ist ein Wortpaar, das häufiger gemeinsam vorkommt, als der Zufall erwarten ließe: *Panama Papers*, *Buenos Aires*, *FC Bayern*. Wer sie findet, gewinnt Mehrwortausdrücke zurück, die von Einzeltermen zerlegt werden.

Das klassische Maß ist die **Pointwise Mutual Information (PMI)**: wie viel häufiger das Paar vorkommt, als es vorkäme, wenn die beiden Wörter unabhängig wären,

$$\text{PMI}(x, y) = \log_2 \frac{P(x, y)}{P(x)\,P(y)}.$$

In [ ]:
# Berechne die PMI für alle benachbarten Wortpaare, die mindestens min_pair Mal vorkommen
def collocations(token_lists, min_pair=3) -> pd.DataFrame:
    unigrams = Counter(t for toks in token_lists for t in toks)
    pairs = Counter(p for toks in token_lists for p in zip(toks, toks[1:]))
    n_uni, n_pair = sum(unigrams.values()), sum(pairs.values())
    rows = []
    for (x, y), n in pairs.items():
        if n < min_pair:
            continue
        pmi = math.log2((n / n_pair) / ((unigrams[x] / n_uni) * (unigrams[y] / n_uni)))
        rows.append((f"{x} {y}", n, unigrams[x], unigrams[y], pmi))
    return pd.DataFrame(rows, columns=["pair", "n_pair", "n_x", "n_y", "PMI"])


# Zeige die Paare mit der höchsten PMI und die häufigsten Paare nebeneinander
coll = collocations(corpus["lemmas"].tolist())
pd.concat(
    {
        "höchste PMI": coll.nlargest(12, "PMI").reset_index(drop=True),
        "am häufigsten": coll.nlargest(12, "n_pair").reset_index(drop=True),
    },
    axis=1,
).round(1)

PMI hat eine bekannte Verzerrung: Sie **bevorzugt seltene Paare**. Zwei Wörter, die jeweils dreimal vorkommen, und zwar immer gemeinsam, erreichen die größtmögliche PMI - ein perfekter Zusammenhang, aber auf fast keiner Evidenz. Die Schwelle `min_pair` ist das übliche Gegenmittel und der Grund, warum es sie gibt. Wer sie erhöht, tauscht Vollständigkeit gegen Verlässlichkeit; häufigkeitsgewichtete Maße wie die Log-Likelihood (aus dem Abschnitt "Keyness", angewendet auf Paare) sind die Alternative.

In der linken Spalte sieht man die Verzerrung direkt: Die Paare mit der höchsten PMI kommen alle genau so oft vor wie die Schwelle verlangt, und ihre Wörter kommen nirgends sonst vor - Film- und Serientitel, Personen, Städte. Die beiden Spalten zeigen Komplementäres: Eine hohe PMI findet feste Namen und Fachbegriffe, eine hohe Häufigkeit findet die wiederkehrenden Geschichten des Korpus.

## Konkordanzen: zurück zum Text

Jede Statistik oben reduziert Text auf Zahlen. Bevor wir einer davon vertrauen, lesen wir den Term im Kontext. Eine **Konkordanz** oder *Keyword in Context* (KWIC) stellt jedes Vorkommen eines Terms mit seiner Umgebung untereinander dar.

In [ ]:
def kwic(term: str, frame: pd.DataFrame = news, width: int = 45, n: int = 12) -> None:
    """Gib die Vorkommen eines Lemmas zentriert aus, mit dem Namen der Zeitung.

    Gesucht wird nach Lemmata, daher werden alle flektierten Formen gefunden; angezeigt
    wird der Originaltext, der über den Zeichen-Offset jedes Tokens gefunden wird.
    """
    shown = 0
    for outlet, doc_id in zip(frame["outlet"], frame["doc_id"]):
        d = docs[doc_id]
        for t in d:
            if t.lemma_.casefold() == term:
                left = d.text[: t.idx][-width:]
                right = d.text[t.idx + len(t.text):][:width]
                print(f"{outlet:>19}  {left:>{width}} [{t.text}] {right}")
                shown += 1
                if shown >= n:
                    return


# Zeige die Konkordanz für den stärksten Schlüsselbegriff von Der Standard
top_term = key.nlargest(1, "G2")["term"].iloc[0]
print(f"Konkordanz für den stärksten Schlüsselbegriff von Der Standard: {top_term!r}\n")
kwic(top_term)

Schon ein Dutzend Zeilen zu lesen zeigt meist, zu welcher der Arten ein Schlüsselbegriff gehört. Hier ist es eindeutig Innenpolitik: Die SPÖ erscheint in Landtagswahlen, Koalitionsfragen und Parteiaustritten, fast immer im Verhältnis zur FPÖ - das beherrschende innenpolitische Thema in Österreich in diesem Jahr. Ein Wort, das nach einem Thema aussieht, kann sich in der Konkordanz aber auch als Format entpuppen; probiere es mit `kwic("presseschau")`. Mach das zur Gewohnheit: **Jede Zahl bekommt eine Konkordanz.**

## Begriffe im Zeitverlauf

Das Korpus umfasst ein Jahr, und Nachrichten kommen in Schüben (engl. *bursts*): Ein Thema taucht auf, erreicht einen Höhepunkt und verschwindet wieder. Zählen wir einen Term je Zeitabschnitt, sehen wir seinen Verlauf; vergleichen wir die Länder, sehen wir, wer wann darüber berichtet hat.

Der Zeitabschnitt ist hier der **Monat**, weil die SZ-Daten das Erscheinungsdatum nur auf den Monat genau kennen (Skript 01). Das ist gröber, als Nachrichten verlaufen - ein Ereignis, das drei Tage lang die Schlagzeilen bestimmt, verschwindet in einem Monat mit tausend anderen Meldungen -, reicht aber, um die großen Geschichten des Jahres zu sehen.

Um schubartig auftretende Terme zu finden, ohne sie von Hand auszuwählen, bewerten wir jeden Term danach, wie konzentriert seine Vorkommen sind: der Anteil seiner Schlagzeilen, der auf seinen stärksten Monat fällt. Bei zwölf Monaten läge dieser Anteil für einen gleichmäßig verteilten Term bei etwa einem Zwölftel. Zwei Einschränkungen halten den Wert aussagekräftig. Wir bleiben bei den Politik-Ressorts, weil etwa im Sport Saisonzeiten (Bundesliga-Pause, Olympische Spiele, Fußball-EM) jeden Vereinsnamen zu einem "Schub" machen würden. Und wir verwenden nur Substantive und Eigennamen, weil Ereignisse mit Substantiven benannt werden - ein Verb oder Adverb, das in einem Monat gehäuft auftritt, ist meist Zufall.

In [ ]:
# Bestimme die Substantive jeder Schlagzeile (der Monat steht schon in der Spalte "month")
news["nouns"] = [
    [t.lemma_.casefold() for t in docs[i] if t.pos_ in {"NOUN", "PROPN"} and is_content(t)]
    for i in news["doc_id"]
]
# Eine Zeile je (Dokument, Term): Ein Term zählt einmal je Schlagzeile, wie oft er dort
# auch vorkommt. Wichtig ist das Entfernen von Duplikaten über doc_id - würden wir
# stattdessen doppelte (Monat, Term)-Zeilen entfernen, flössen verschiedene Schlagzeilen
# desselben Monats zusammen.
doc_terms = news[["doc_id", "month", "country", "nouns"]].explode("nouns")
doc_terms = doc_terms.rename(columns={"nouns": "term"}).dropna().drop_duplicates(["doc_id", "term"])

# Berechne je Term den Anteil der Schlagzeilen im stärksten Monat (nur Terme mit mindestens 40 Schlagzeilen)
per_month = doc_terms.groupby(["term", "month"]).size()
totals = per_month.groupby("term").sum()
burst = (per_month.groupby("term").max() / totals)[totals >= 40].sort_values(ascending=False)
bursty = burst.head(6).index.tolist()
burst.head(6).to_frame("Anteil im stärksten Monat")

In [ ]:
# Zeichne für jeden der sechs Terme die Schlagzeilen pro Monat, gestapelt nach Land
months = pd.date_range(news["month"].min(), news["month"].max(), freq="MS")
fig, axes = plt.subplots(2, 3, figsize=(11, 5), sharex=True)
for ax, term in zip(axes.ravel(), bursty):
    bottom = np.zeros(len(months))
    for country, outlet in [("AT", "Der Standard"), ("DE", "Süddeutsche Zeitung")]:
        s = doc_terms[(doc_terms["term"] == term) & (doc_terms["country"] == country)].groupby("month").size()
        values = s.reindex(months, fill_value=0).values
        ax.bar(range(len(months)), values, bottom=bottom, color=tm.OUTLET_COLOURS[outlet], label=outlet)
        bottom += values
    ax.set_title(term, fontsize=10)
    ax.set_xticks(range(0, len(months), 2))
    ax.set_xticklabels([m.strftime("%m/%y") for m in months[::2]], fontsize=8)
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Schubartige Terme: Schlagzeilen pro Monat (Politik-Ressorts)", fontweight="bold")
fig.tight_layout()
plt.show()

Die meisten Schübe markieren Ereignisse des Jahres: die *Panama Papers* im April 2016, die Anschläge von *Paris* im November 2015 und von *Brüssel* im März 2016, *Ungarn* im Spätsommer 2015, als die Flüchtlinge auf dem Weg nach Westen an der ungarischen Grenze und am Budapester Bahnhof festsaßen. Der Vergleich der beiden Farben zeigt, ob eine Geschichte in beiden Ländern in den Politik-Ressorts erzählt wurde. Bei den Panama Papers fehlt Der Standard fast ganz - nicht weil er nicht darüber berichtet hätte, sondern weil er sie im Ressort *Wirtschaft* führt (siehe oben).

Lies die Balken als Anteile, nicht als Mengen: Die SZ liefert in jedem Monat mehr als dreimal so viele Politik-Schlagzeilen wie Der Standard, ihre Farbe dominiert also jeden gemeinsamen Schub.

Nicht jeder schubartige Term ist ein einzelnes Ereignis. *Spanien* etwa verteilt sich auf eine Parlamentswahl, Festnahmen mutmaßlicher Terroristen und Regierungsbildung über mehrere Monate; der Wert wird mechanisch berechnet, und auch ein Wort, das sich zufällig in einem Monat häuft, kann weit oben landen. Wie immer gilt: Prüfe den stärksten Monat mit `kwic`, bevor du etwas eine Geschichte nennst.

## Zusammenfassung

| Methode | Beantwortet | Achte auf |
|---|---|---|
| Dokument-Term-Matrix | die gemeinsame Grundlage für alles andere | sie verwirft die Wortreihenfolge |
| TF-IDF | was für ein Dokument charakteristisch ist | eine Gewichtung, kein Test |
| Log-Likelihood (G²) | ist ein Unterschied echt? | wächst mit der Häufigkeit, nicht nur mit dem Effekt |
| Log Ratio | wie groß ist ein Unterschied? | instabil bei seltenen Termen |
| PMI | welche Wörter gehören zusammen? | bevorzugt seltene Paare |
| Konkordanz | was bedeutet der Term hier tatsächlich? | sie ist langsam - genau darum geht es |

## Übungen

1. **Die Wirtschaftsressorts.** Wiederhole die Keyness-Analyse für die Ressortgruppe `Wirtschaft`. Hält das nationale Muster aus der Politik, oder treten andere Arten von Schlüsselbegriffen in den Vordergrund? Wiederhole es für `Sport`: Was sagen die Schlüsselbegriffe dort über die beiden Länder?
2. **Noch einmal Dachzeilen.** Entferne die Dachzeilen (Kicker) beider Zeitungen (Skript 01: alles bis zum ersten Doppelpunkt oder `" - "`) und führe die Keyness-Analyse erneut durch. Welche Schlüsselbegriffe verschwinden?
3. **Stärke von Kollokationen.** Schreibe `collocations` so um, dass Paare mit der Log-Likelihood statt mit der PMI bewertet werden. Wie verändern sich die obersten 20?
4. **Dein eigener Schub.** Wähle einen Begriff, an den du dich aus den Nachrichten von 2015/16 erinnerst (etwa *Brexit*, *Köln*, *Hofburg* oder *Idomeni*), und zeichne ihn pro Monat für das ganze Korpus statt nur für die Politik-Ressorts. Prüfe die stärksten Monate mit `kwic`. Für Der Standard kannst du den Verlauf auch pro Tag zeichnen (Spalte `published`) - wie viel geht in der Monatsansicht verloren?

## Quellen

* https://spacy.io/models/de
* https://spacy.io/usage/linguistic-features
* https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
* https://de.wikipedia.org/wiki/Tf-idf-Ma%C3%9F
* https://pandas.pydata.org/docs/
* https://matplotlib.org/